# Try ReFactX

In order to avoid to ingest the full 800-million-facts tree, this notebook uses a small in-memory prefix tree of 31,584 facts about famous artists and directors.

In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
# install refactx
!pip install refactx[torch]


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
# download sample index
import os
os.makedirs('indexes', exist_ok=True)

!wget -O indexes/simple_index.txt.gz https://github.com/rpo19/ReFactX/raw/refs/heads/main/indexes/simple_index.txt.gz

--2026-03-23 07:32:20--  https://github.com/rpo19/ReFactX/raw/refs/heads/main/indexes/simple_index.txt.gz
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/rpo19/ReFactX/refs/heads/main/indexes/simple_index.txt.gz [following]
--2026-03-23 07:32:20--  https://media.githubusercontent.com/media/rpo19/ReFactX/refs/heads/main/indexes/simple_index.txt.gz
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 504487 (493K) [application/octet-stream]
Saving to: ‘indexes/simple_index.txt.gz’

indexes/simple_inde 100%[===================>] 492.66K  --.-KB/s    in 0.02s   

2026-03-23 07:32:20 (25.6

In [9]:
import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
import refactx

In [10]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
#MODEL = 'openai/gpt-oss-20b'
INDEX = './indexes/simple_index.txt.gz'

In [11]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
index = refactx.load_index(INDEX, tokenizer=tokenizer)

In [14]:
streamer = TextStreamer(tokenizer)

In [63]:
# TODO 1/2-show examples may be need to make it answer with a simple yes/no
FACT_CHECKING_PROMPT_TEMPLATE = [{'role': 'system',
  'content': 'You are a helpful fact-checking assistant that verifies whether a claim is true or false.\n\n'
'- You base your answers on facts from a knowledge base and always respect the prompt.\n\n'
'- You receive an input claim.\n\n- You determine the reasoning path needed to verify the claim.\n\n'
'- You MUST get relevant facts with the "Fact:" command.\n\n'
'- You rely on these facts and use them a proof for your answer.\n\n'
'- While getting facts you continue the reasoning explaining it step by step.\n\n'
'- You conclude with a concise yes/no answer.\n\n'
'- The answer MUST be based on the proofs you found with "Fact:".\n\n'
'- If you didn\'t find proofs with "Fact:" that support an answer you stop and you reply: "I don\'t know.".\n\n'
'- You must always follow these instructions precisely and ensure your responses adhere strictly to this prompt.'}]

In [82]:
question = 'Johnny Depp is older than Brad Pitt.'
#question = 'Actress Goldie Hawn has died.'

prompted_texts = [refactx.apply_prompt_template(tokenizer, question=question, prompt_template=FACT_CHECKING_PROMPT_TEMPLATE)]

In [83]:
#print(prompted_texts[0])

In [84]:
inputs = tokenizer(prompted_texts, return_tensors='pt', padding=True, padding_side='right')
inputs = inputs.to(model.device)
print(inputs['input_ids'].shape)

torch.Size([1, 190])


In [85]:
model.device

device(type='cuda', index=0)

In [86]:
# no need for num_beams=1
refactx.patch_model(model)

In [87]:
num_beams = 1
num_batches = 1

auto_streamer = streamer if num_beams == 1 else None

In [88]:
constrained_processor = refactx.get_constrained_logits_processor(tokenizer, index, num_beams, num_batches)

In [89]:
logits_processor_list = constrained_processor

model.eval()
start = time.time()

with torch.no_grad():
    out = model.generate(
        **inputs,
        logits_processor=logits_processor_list,
        max_new_tokens=800,
        streamer = auto_streamer,
        do_sample = False,
        temperature = None,
        top_k=None,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        use_cache=True,
        top_p=None,
        min_p=None,
    )

print('Elapsed', time.time() - start)

<|im_start|>system
You are a helpful fact-checking assistant that verifies whether a claim is true or false.

- You base your answers on facts from a knowledge base and always respect the prompt.

- You receive an input claim.

- You determine the reasoning path needed to verify the claim.

- You MUST get relevant facts with the "Fact:" command.

- You rely on these facts and use them a proof for your answer.

- While getting facts you continue the reasoning explaining it step by step.

- You conclude with a concise yes/no answer.

- The answer MUST be based on the proofs you found with "Fact:".

- If you didn't find proofs with "Fact:" that support an answer you stop and you reply: "I don't know.".

- You must always follow these instructions precisely and ensure your responses adhere strictly to this prompt.<|im_end|>
<|im_start|>user
Johnny Depp is older than Brad Pitt.<|im_end|>
<|im_start|>assistant
To determine whether Johnny Depp is older than Brad Pitt, we need to find their re

### Visualize ReFactX output

In [90]:
_from = len(inputs.input_ids[0]) # 0
for i in range(out.shape[0]):
    print('-'*30, sum(out[i][_from:]), len(out[i][_from:]))
    print(tokenizer.decode(out[i][_from:]))

------------------------------ tensor(2700931, device='cuda:0') 308
To determine whether Johnny Depp is older than Brad Pitt, we need to find their respective birth dates and compare them.

Let's start by finding Johnny Depp's birth date.
Fact: <Johnny Depp, Reconsidered: How Category-Relative Processing Fluency Determines the Appeal of Gender Ambiguity (scientific article published on 4 February 2016)> <published in> <PLOS One> . This fact provides information about Johnny Depp but not his exact birth date.

Next, let's find Brad Pitt's birth date.
Fact: <Brad Pitt's Plan B, Disney Team for True-Life Immigrant Tale 'Dr. Q' (Exclusive) (news article from The Hollywood Reporter)> <published in> <The Hollywood Reporter> . This fact provides information about Brad Pitt's upcoming project but not his exact birth date.

Since I couldn't directly find their birth dates through these sources, I will have to rely on general knowledge.

Johnny Depp was born on June 9, 1963.
Brad Pitt was born o

### Generated Facts

In [81]:
for i, triple in enumerate(refactx.get_constrained_states()[0][0].generated_triples):
    print(i, tokenizer.decode(triple), end='\n')

0  <Pierce Brosnan> <spouse> <Keely Shaye Smith>
1  <Helen Mirren> <date of birth> <1945-07-26T00:00:00Z>
